# Notebook 10 — Signer-Independent (SI) Evaluation: ST-GCN 5-Fold CV

**Milestone 3 — RQ2: How does signer-independent evaluation degrade performance relative to signer-dependent evaluation?**

### Protocol
- **5-fold cross-validation** using proxy signer groups (3 groups held out per fold).
- **⚠ PROVISIONAL — proxy signer IDs:** The INCLUDE HuggingFace distribution does not
  ship real signer identity labels. Groups were assigned by ranking videos within each
  class by MVI filename (`rank % 15`). These do NOT correspond to real individuals
  (confirmed in NB02: uneven per-group counts, 387→125 vs ~286 expected). Results **must
  not be reported as true cross-signer generalisation** — they approximate the SI setting
  and establish an upper-bound estimate of generalisation difficulty.
- Results reported as **mean ± std across 5 folds**.
- Per the annotated INCLUDE paper (Paper04): no prior work has reported leave-signers-out
  evaluation on INCLUDE — this remains a novel contribution even under the proxy protocol.

### Model
- **Adaptive ST-GCN** (53-joint, learnable B, T=32, full augmentation, torso norm)
- Training: SGD + Nesterov (lr=0.01, momentum=0.9), cosine LR, weight_decay=1e-4,
  batch=32, max 80 epochs, patience=10, SEED=42.
- **Validation carve-out**: 10% of each fold's training set, stratified by class (same as NB04).

### Sections
1. Setup and data loading
2. Per-fold training (adaptive ST-GCN)
3. Results summary (mean ± std)
4. SD vs SI gap analysis
5. Save to `results/metrics_all.csv` and `results/ablation_SI_stgcn.csv`

In [1]:
import sys, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

PROJECT_ROOT = Path("/Users/yamini/Desktop/projects/ISL PROJECT")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from train import PROC_DIR, N_CLASSES, get_device, set_seed, build_adjacency
from dataset import SkeletonDataset
from model import STGCN

DEVICE    = get_device()
SEED      = 42
T_SI      = 32       # temporal window (best from A1)
BATCH     = 32
MAX_EP    = 80
PATIENCE  = 10
VAL_FRAC  = 0.10     # fraction of train carved out as validation
CKPT_DIR  = PROJECT_ROOT / "checkpoints"
RES_DIR   = PROJECT_ROOT / "results"

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"N_CLASSES: {N_CLASSES}")
print(f"T={T_SI}  batch={BATCH}  max_ep={MAX_EP}  patience={PATIENCE}")

Device: mps
N_CLASSES: 262
T=32  batch=32  max_ep=80  patience=10


In [2]:
# ── Load SI splits ────────────────────────────────────────────────────────────
with open(PROC_DIR / "si_splits.pkl", "rb") as f:
    si_splits = pickle.load(f)

print(f"Loaded {len(si_splits)} folds")
for i, fold in enumerate(si_splits):
    print(f"  Fold {i} | held_out groups={fold['held_out']} "
          f"| train={len(fold['X_train'])}  test={len(fold['X_test'])}")
    print(f"          | X_train shape: {fold['X_train'].shape}  "
          f"(will resample T: {fold['X_train'].shape[1]} → {T_SI})")

Loaded 5 folds
  Fold 0 | held_out groups=[0, 1, 2] | train=2497  test=1155
          | X_train shape: (2497, 64, 53, 3)  (will resample T: 64 → 32)
  Fold 1 | held_out groups=[3, 4, 5] | train=2657  test=995
          | X_train shape: (2657, 64, 53, 3)  (will resample T: 64 → 32)
  Fold 2 | held_out groups=[6, 7, 8] | train=3005  test=647
          | X_train shape: (3005, 64, 53, 3)  (will resample T: 64 → 32)
  Fold 3 | held_out groups=[9, 10, 11] | train=3172  test=480
          | X_train shape: (3172, 64, 53, 3)  (will resample T: 64 → 32)
  Fold 4 | held_out groups=[12, 13, 14] | train=3277  test=375
          | X_train shape: (3277, 64, 53, 3)  (will resample T: 64 → 32)


---
## Section 2 — Per-Fold Training (Adaptive ST-GCN)

In [3]:
# ── Training function (matches NB08/NB09 hyperparameters exactly) ─────────────
def train_si_fold(fold_idx: int, fold_dict: dict) -> dict:
    """
    Train adaptive ST-GCN on one SI fold.
    Carves VAL_FRAC of training samples for early stopping.
    Saves best checkpoint to checkpoints/adaptive_T32_full_SI_fold{i}.pt.
    Returns dict with val_acc, test_acc, macro_f1.
    """
    set_seed(SEED)
    t0 = time.time()

    X_all = fold_dict["X_train"]   # (N, 64, 53, 3)
    y_all = fold_dict["y_train"]
    X_te  = fold_dict["X_test"]
    y_te  = fold_dict["y_test"]

    # Carve validation set (fixed random permutation, seeded)
    rng   = np.random.RandomState(SEED)
    idx   = rng.permutation(len(X_all))
    n_val = max(1, int(VAL_FRAC * len(X_all)))
    X_tr, y_tr = X_all[idx[n_val:]], y_all[idx[n_val:]]
    X_va, y_va = X_all[idx[:n_val]], y_all[idx[:n_val]]

    # Datasets — resample to T=32, full augmentation on train only
    tr_ds = SkeletonDataset(X_tr, y_tr, augment=False, resample_T=T_SI)  # FIXED 2026-08-23: was augment=True, mixing configs vs the no-aug SD number (Naman review)
    va_ds = SkeletonDataset(X_va, y_va, augment=False, resample_T=T_SI)
    te_ds = SkeletonDataset(X_te, y_te, augment=False, resample_T=T_SI)

    kw = dict(batch_size=BATCH, num_workers=0, pin_memory=False)
    tr_dl = DataLoader(tr_ds, shuffle=True,  **kw)
    va_dl = DataLoader(va_ds, shuffle=False, **kw)
    te_dl = DataLoader(te_ds, shuffle=False, **kw)

    print(f"\n{'='*62}")
    print(f"  SI Fold {fold_idx}  |  held_out={fold_dict['held_out']}")
    print(f"  train={len(tr_ds)}  val={len(va_ds)}  test={len(te_ds)}")
    print(f"{'='*62}")

    # Build adaptive ST-GCN (same topology as best SD model)
    A = build_adjacency(n_joints=53, strategy="spatial")
    model = STGCN(n_classes=N_CLASSES, A=A, adaptive=True).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {n_params:,}")

    # Optimiser — identical to NB08/NB09
    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.01, momentum=0.9,
        nesterov=True, weight_decay=1e-4
    )
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EP)
    criterion  = nn.CrossEntropyLoss()

    best_val, no_imp = 0.0, 0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    for ep in range(1, MAX_EP + 1):
        # ── train ──────────────────────────────────────────────────────────────
        model.train(); correct = total = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward(); optimizer.step()
            correct += (logits.detach().argmax(1) == yb).sum().item()
            total   += len(yb)

        # ── validate ───────────────────────────────────────────────────────────
        model.eval(); vc = vt = 0
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vc += (model(xb).argmax(1) == yb).sum().item(); vt += len(yb)
        vl_acc = vc / vt
        scheduler.step()

        elapsed = (time.time() - t0) / 60
        print(f"  [Fold {fold_idx}] ep {ep:3d}/{MAX_EP}  "
              f"tr={correct/total:.4f}  vl={vl_acc:.4f}  ({elapsed:.1f}m)",
              flush=True)

        if vl_acc > best_val:
            best_val   = vl_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print(f"  [Fold {fold_idx}] Early stop at epoch {ep}"); break

    # ── test ───────────────────────────────────────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in te_dl:
            xb = xb.to(DEVICE)
            preds = model(xb).argmax(1).cpu()
            y_true.extend(yb.numpy()); y_pred.extend(preds.numpy())
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    test_acc = (y_true == y_pred).mean()
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    elapsed = (time.time() - t0) / 60
    print(f"\n  [Fold {fold_idx}] DONE  val={best_val:.4f}  "
          f"test={test_acc:.4f}  F1={macro_f1:.4f}  ({elapsed:.1f}m)")

    # ── save checkpoint ────────────────────────────────────────────────────────
    ckpt_path = CKPT_DIR / f"adaptive_T32_none_SI_fold{fold_idx}.pt"
    torch.save({
        "fold":         fold_idx,
        "held_out":     fold_dict["held_out"],
        "topology":     "adaptive",
        "T":            T_SI,
        "augmentation": "none",
        "normalisation":"torso",
        "protocol":     "SI-proxy",
        "val_acc":      best_val,
        "test_acc":     test_acc,
        "macro_f1":     macro_f1,
        "state_dict":   best_state,
    }, ckpt_path)
    print(f"  Saved {ckpt_path.name}")

    return dict(
        fold=fold_idx, held_out=str(fold_dict["held_out"]),
        n_train=len(tr_ds), n_val=len(va_ds), n_test=len(te_ds),
        val_acc=best_val, test_acc=test_acc, macro_f1=macro_f1,
    )

print("train_si_fold() defined — ready to run.")

train_si_fold() defined — ready to run.


In [4]:
# ── Fold 0 ────────────────────────────────────────────────────────────────────
result_fold0 = train_si_fold(0, si_splits[0])


  SI Fold 0  |  held_out=[0, 1, 2]
  train=2248  val=249  test=1155
  Params: 2,495,370
  [Fold 0] ep   1/80  tr=0.0067  vl=0.0080  (0.4m)
  [Fold 0] ep   2/80  tr=0.0129  vl=0.0281  (0.7m)
  [Fold 0] ep   3/80  tr=0.0209  vl=0.0080  (1.0m)
  [Fold 0] ep   4/80  tr=0.0289  vl=0.0281  (1.3m)
  [Fold 0] ep   5/80  tr=0.0396  vl=0.0281  (1.5m)
  [Fold 0] ep   6/80  tr=0.0498  vl=0.0562  (1.8m)
  [Fold 0] ep   7/80  tr=0.0583  vl=0.0402  (2.0m)
  [Fold 0] ep   8/80  tr=0.0707  vl=0.0321  (2.3m)
  [Fold 0] ep   9/80  tr=0.0752  vl=0.0763  (2.5m)
  [Fold 0] ep  10/80  tr=0.1174  vl=0.0843  (2.8m)
  [Fold 0] ep  11/80  tr=0.1299  vl=0.0723  (3.0m)
  [Fold 0] ep  12/80  tr=0.1428  vl=0.0843  (3.3m)
  [Fold 0] ep  13/80  tr=0.1624  vl=0.1165  (3.5m)
  [Fold 0] ep  14/80  tr=0.1779  vl=0.1446  (3.8m)
  [Fold 0] ep  15/80  tr=0.2246  vl=0.2048  (4.0m)
  [Fold 0] ep  16/80  tr=0.2180  vl=0.1928  (4.3m)
  [Fold 0] ep  17/80  tr=0.2682  vl=0.2691  (4.7m)
  [Fold 0] ep  18/80  tr=0.2714  vl=0.3133  

In [5]:
# ── Fold 1 ────────────────────────────────────────────────────────────────────
result_fold1 = train_si_fold(1, si_splits[1])


  SI Fold 1  |  held_out=[3, 4, 5]
  train=2392  val=265  test=995
  Params: 2,495,370
  [Fold 1] ep   1/80  tr=0.0033  vl=0.0038  (0.2m)
  [Fold 1] ep   2/80  tr=0.0100  vl=0.0113  (0.4m)
  [Fold 1] ep   3/80  tr=0.0201  vl=0.0113  (0.6m)
  [Fold 1] ep   4/80  tr=0.0322  vl=0.0528  (0.8m)
  [Fold 1] ep   5/80  tr=0.0439  vl=0.0453  (1.0m)
  [Fold 1] ep   6/80  tr=0.0414  vl=0.0491  (1.1m)
  [Fold 1] ep   7/80  tr=0.0548  vl=0.0302  (1.3m)
  [Fold 1] ep   8/80  tr=0.0707  vl=0.0453  (1.5m)
  [Fold 1] ep   9/80  tr=0.0882  vl=0.0792  (1.7m)
  [Fold 1] ep  10/80  tr=0.1196  vl=0.0830  (1.9m)
  [Fold 1] ep  11/80  tr=0.1267  vl=0.1245  (2.1m)
  [Fold 1] ep  12/80  tr=0.1518  vl=0.1057  (2.2m)
  [Fold 1] ep  13/80  tr=0.1689  vl=0.1321  (2.4m)
  [Fold 1] ep  14/80  tr=0.1877  vl=0.2000  (2.6m)
  [Fold 1] ep  15/80  tr=0.2195  vl=0.1509  (2.8m)
  [Fold 1] ep  16/80  tr=0.2358  vl=0.2113  (3.0m)
  [Fold 1] ep  17/80  tr=0.2688  vl=0.1887  (3.2m)
  [Fold 1] ep  18/80  tr=0.2864  vl=0.2906  (

In [6]:
# ── Fold 2 ────────────────────────────────────────────────────────────────────
result_fold2 = train_si_fold(2, si_splits[2])


  SI Fold 2  |  held_out=[6, 7, 8]
  train=2705  val=300  test=647
  Params: 2,495,370
  [Fold 2] ep   1/80  tr=0.0055  vl=0.0067  (0.2m)
  [Fold 2] ep   2/80  tr=0.0240  vl=0.0067  (0.4m)
  [Fold 2] ep   3/80  tr=0.0277  vl=0.0100  (0.6m)
  [Fold 2] ep   4/80  tr=0.0410  vl=0.0267  (0.9m)
  [Fold 2] ep   5/80  tr=0.0384  vl=0.0367  (1.1m)
  [Fold 2] ep   6/80  tr=0.0525  vl=0.0733  (1.3m)
  [Fold 2] ep   7/80  tr=0.0810  vl=0.0667  (1.5m)
  [Fold 2] ep   8/80  tr=0.0998  vl=0.1233  (1.7m)
  [Fold 2] ep   9/80  tr=0.1157  vl=0.1400  (1.9m)
  [Fold 2] ep  10/80  tr=0.1375  vl=0.1500  (2.1m)
  [Fold 2] ep  11/80  tr=0.1590  vl=0.1800  (2.3m)
  [Fold 2] ep  12/80  tr=0.1837  vl=0.1933  (2.6m)
  [Fold 2] ep  13/80  tr=0.2144  vl=0.2033  (2.8m)
  [Fold 2] ep  14/80  tr=0.2421  vl=0.2400  (3.0m)
  [Fold 2] ep  15/80  tr=0.2636  vl=0.2233  (3.2m)
  [Fold 2] ep  16/80  tr=0.2891  vl=0.2900  (3.4m)
  [Fold 2] ep  17/80  tr=0.3006  vl=0.2600  (3.6m)
  [Fold 2] ep  18/80  tr=0.3405  vl=0.2867  (

In [7]:
# ── Fold 3 ────────────────────────────────────────────────────────────────────
result_fold3 = train_si_fold(3, si_splits[3])


  SI Fold 3  |  held_out=[9, 10, 11]
  train=2855  val=317  test=480
  Params: 2,495,370
  [Fold 3] ep   1/80  tr=0.0042  vl=0.0095  (0.2m)
  [Fold 3] ep   2/80  tr=0.0147  vl=0.0315  (0.5m)
  [Fold 3] ep   3/80  tr=0.0214  vl=0.0189  (0.7m)
  [Fold 3] ep   4/80  tr=0.0315  vl=0.0410  (0.9m)
  [Fold 3] ep   5/80  tr=0.0403  vl=0.0789  (1.1m)
  [Fold 3] ep   6/80  tr=0.0494  vl=0.1009  (1.3m)
  [Fold 3] ep   7/80  tr=0.0718  vl=0.0631  (1.6m)
  [Fold 3] ep   8/80  tr=0.0904  vl=0.1104  (1.8m)
  [Fold 3] ep   9/80  tr=0.1145  vl=0.1167  (2.0m)
  [Fold 3] ep  10/80  tr=0.1482  vl=0.1388  (2.2m)
  [Fold 3] ep  11/80  tr=0.1541  vl=0.1388  (2.4m)
  [Fold 3] ep  12/80  tr=0.1825  vl=0.1735  (2.7m)
  [Fold 3] ep  13/80  tr=0.2126  vl=0.2650  (2.9m)
  [Fold 3] ep  14/80  tr=0.2389  vl=0.2555  (3.1m)
  [Fold 3] ep  15/80  tr=0.2532  vl=0.3344  (3.4m)
  [Fold 3] ep  16/80  tr=0.2967  vl=0.3123  (3.6m)
  [Fold 3] ep  17/80  tr=0.3163  vl=0.3596  (3.8m)
  [Fold 3] ep  18/80  tr=0.3401  vl=0.3754 

In [8]:
# ── Fold 4 ────────────────────────────────────────────────────────────────────
result_fold4 = train_si_fold(4, si_splits[4])


  SI Fold 4  |  held_out=[12, 13, 14]
  train=2950  val=327  test=375
  Params: 2,495,370
  [Fold 4] ep   1/80  tr=0.0047  vl=0.0061  (0.2m)
  [Fold 4] ep   2/80  tr=0.0197  vl=0.0061  (0.5m)
  [Fold 4] ep   3/80  tr=0.0220  vl=0.0153  (0.7m)
  [Fold 4] ep   4/80  tr=0.0275  vl=0.0214  (0.9m)
  [Fold 4] ep   5/80  tr=0.0336  vl=0.0367  (1.1m)
  [Fold 4] ep   6/80  tr=0.0495  vl=0.0428  (1.4m)
  [Fold 4] ep   7/80  tr=0.0678  vl=0.0581  (1.6m)
  [Fold 4] ep   8/80  tr=0.0868  vl=0.0489  (1.8m)
  [Fold 4] ep   9/80  tr=0.0953  vl=0.1254  (2.1m)
  [Fold 4] ep  10/80  tr=0.1190  vl=0.1101  (2.3m)
  [Fold 4] ep  11/80  tr=0.1468  vl=0.1284  (2.5m)
  [Fold 4] ep  12/80  tr=0.1644  vl=0.1407  (2.7m)
  [Fold 4] ep  13/80  tr=0.1759  vl=0.1804  (3.0m)
  [Fold 4] ep  14/80  tr=0.2156  vl=0.1927  (3.2m)
  [Fold 4] ep  15/80  tr=0.2346  vl=0.2385  (3.4m)
  [Fold 4] ep  16/80  tr=0.2627  vl=0.1927  (3.6m)
  [Fold 4] ep  17/80  tr=0.2814  vl=0.2141  (3.9m)
  [Fold 4] ep  18/80  tr=0.3058  vl=0.3119

---
## Section 3 — Results Summary

In [9]:
# ── Collect and summarise all fold results ────────────────────────────────────
# Reloads from saved checkpoints so this cell is safe to re-run independently
fold_rows = []
for i in range(5):
    ckpt_path = CKPT_DIR / f"adaptive_T32_none_SI_fold{i}.pt"
    if not ckpt_path.exists():
        print(f"  MISSING: {ckpt_path.name} — run fold {i} cell first")
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    fold_rows.append(dict(
        fold=i,
        held_out=str(ckpt["held_out"]),
        val_acc=ckpt["val_acc"],
        test_acc=ckpt["test_acc"],
        macro_f1=ckpt["macro_f1"],
    ))

df_folds = pd.DataFrame(fold_rows)

print("=" * 65)
print("ST-GCN SI-proxy 5-Fold Results  (adaptive, T=32, NO aug, torso -- fixed-config)")
print("⚠  PROVISIONAL — proxy signer groups, not verified signer IDs")
print("=" * 65)
print(df_folds.to_string(index=False))

if len(df_folds) == 5:
    top1_mean = df_folds["test_acc"].mean()
    top1_std  = df_folds["test_acc"].std()
    f1_mean   = df_folds["macro_f1"].mean()
    f1_std    = df_folds["macro_f1"].std()
    print(f"\n  Top-1  : {top1_mean:.4f} ± {top1_std:.4f}")
    print(f"  Macro-F1: {f1_mean:.4f} ± {f1_std:.4f}")
else:
    print(f"\n  Only {len(df_folds)}/5 folds complete — run remaining fold cells.")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

---
## Section 4 — SD vs SI Gap Analysis (RQ2)

In [ ]:
# -- SD vs SI gap -- ONE FIXED CONFIGURATION (augmentation=none) --------------
# FIXED 2026-08-23 (Naman review): the old version compared an SD "no-aug"
# number against an SI-proxy run trained with FULL augmentation, then reported
# both a "gap_none" and "gap_full" as if that were informative -- mixing two
# different training configs is not a valid comparison. Both sides now use
# augmentation=none, same topology/T/normalisation, so this is apples-to-apples.
_fold_rows = []
for _i in range(5):
    _p = CKPT_DIR / f"adaptive_T32_none_SI_fold{_i}.pt"
    if _p.exists():
        _c = torch.load(_p, map_location="cpu", weights_only=False)
        _fold_rows.append(dict(fold=_i, test_acc=_c["test_acc"], macro_f1=_c["macro_f1"]))
df_folds_gap = pd.DataFrame(_fold_rows)

df_metrics = pd.read_csv(RES_DIR / "metrics_all.csv")
sd_none = df_metrics[
    (df_metrics["experiment"]=="ablation") &
    (df_metrics["condition"]=="adaptive_none_torso")
].iloc[0]

if len(df_folds_gap) == 5:
    top1_mean = df_folds_gap["test_acc"].mean()
    top1_std  = df_folds_gap["test_acc"].std()
    f1_mean   = df_folds_gap["macro_f1"].mean()
    f1_std    = df_folds_gap["macro_f1"].std()
    gap       = float(sd_none["test_acc"]) - top1_mean
    print("=" * 65)
    print("RQ2 -- SD vs recording-session-proxy Gap (adaptive ST-GCN, T=32, augmentation=none, fixed config)")
    print("=" * 65)
    print(f"  SD (no aug)                            test_acc = {float(sd_none['test_acc']):.4f}")
    print(f"  Recording-session proxy (5-fold mean)  test_acc = {top1_mean:.4f} +/- {top1_std:.4f}")
    print(f"\n  Gap (SD - proxy) = {gap:+.4f} ({gap*100:+.2f} pp)")
    print("\n  NOTE: always call this 'recording-session proxy' in the paper, never signer-independent --")
    print("  proxy IDs are rank-by-filename, not verified real signer identities.")
else:
    print(f"Only {len(df_folds_gap)}/5 folds complete -- run remaining fold cells first.")


---
## Section 5 — Save Results

In [ ]:
# -- Save per-fold CSV ---------------------------------------------------------
# Reloads from checkpoints -- safe to run without nb10-summary
_fold_rows = []
for _i in range(5):
    _p = CKPT_DIR / f"adaptive_T32_none_SI_fold{_i}.pt"
    if _p.exists():
        _c = torch.load(_p, map_location="cpu", weights_only=False)
        _fold_rows.append(dict(
            fold=_i, held_out=str(_c["held_out"]),
            val_acc=_c["val_acc"], test_acc=_c["test_acc"], macro_f1=_c["macro_f1"],
        ))
df_folds_save = pd.DataFrame(_fold_rows)

if len(df_folds_save) == 5:
    df_folds_save.to_csv(RES_DIR / "ablation_SI_stgcn_fixed.csv", index=False)
    print("Saved results/ablation_SI_stgcn_fixed.csv (old full-aug results/ablation_SI_stgcn.csv kept as documented-invalid record)")

    df_m = pd.read_csv(RES_DIR / "metrics_all.csv")
    df_m = df_m[~((df_m["experiment"]=="SI") &
                  (df_m["condition"].str.startswith("adaptive_none")))].copy()

    si_rows = []
    for _, r in df_folds_save.iterrows():
        si_rows.append(dict(experiment="SI",
                           condition=f"adaptive_none_torso_fold{int(r['fold'])}",
                           T=T_SI, val_acc=round(r["val_acc"],4),
                           test_acc=round(r["test_acc"],4), macro_f1=round(r["macro_f1"],4)))
    si_rows.append(dict(experiment="SI", condition="adaptive_none_torso_mean5fold",
                        T=T_SI,
                        val_acc=round(df_folds_save["val_acc"].mean(),4),
                        test_acc=round(df_folds_save["test_acc"].mean(),4),
                        macro_f1=round(df_folds_save["macro_f1"].mean(),4)))

    df_m = pd.concat([df_m, pd.DataFrame(si_rows)], ignore_index=True)
    df_m.to_csv(RES_DIR / "metrics_all.csv", index=False)
    print("Updated results/metrics_all.csv with SI rows (old full-aug SI rows kept as documented-invalid record)")

    print("\n" + "=" * 45)
    print("FINAL recording-session-proxy RESULT (for M3 submission)")
    print("  ST-GCN adaptive, T=32, NO aug, torso norm (fixed config, matches SD)")
    print("  Protocol: recording-session-proxy 5-fold CV  -- PROVISIONAL, never call signer-independent")
    print(f"  Top-1:    {df_folds_save['test_acc'].mean()*100:.2f}% +/- {df_folds_save['test_acc'].std()*100:.2f}%")
    print(f"  Macro-F1: {df_folds_save['macro_f1'].mean():.4f} +/- {df_folds_save['macro_f1'].std():.4f}")
    print("=" * 45)
else:
    print(f"Only {len(df_folds_save)}/5 folds complete -- run all fold cells before saving.")
